# Merge IRGA72 Level-0 output (2016-2025)

**notebook version**: `2` (31 Jul 2026)

## ℹ️ About this notebook

Merges the EddyPro *FLUXNET* output files of the **Level-0** (preliminary) flux calculations for the
**IRGA72** gas analyser into one continuous 30MIN dataframe and stores it in the external data
folder, as parquet and as csv.

Its task is to bring the per-run output files onto one continuous timestamp index, so that the
Level-0 check notebooks (`03` fluxes, `04` wind direction, `05` time lags) all read the same file
instead of each re-reading the raw output. No value is computed or corrected here. The only change
to the incoming data is that the six biomet columns exported by mistake in the 2025 run are
dropped, which is done here rather than in each reader so that the merged record has one column set
over its whole length.

***

## 📇 Data sources

EddyPro *FLUXNET* output files (`*_adv.csv`), one per processing run, in the external data folder
under `0_data/OPENLAG-IRGA72-Level-0_fluxnet_2016-2025/`.

A run covers one or more **setup periods**, not a calendar year: `2016_2+3` is a single run over
setup periods 2 and 3 of 2016, and a run boundary marks a hardware or configuration change. The
merge therefore works on the timestamps inside the files, not on the file names. The setup periods,
and the numbered notes that go with them, are documented in `docs/Yearly_Notes.md` and on the
SwissFluxNet page it links to.

Three setup periods are missing from this folder, and none of them is an oversight:

- **`2019_3`** (17 Jan - 20 Mar 2019) was measured with the LI-7500 (`IRGA75`), which stood in for
  the defective LI-7200. It is processed with the other `IRGA75` runs and stays in the `IRGA75`
  source folder, which is what the `+2019` in that folder's name means.
- **`2021_2` and `2022_1`** (13 Dec 2021 - 7 Feb 2022) have no output at all: the raw data was
  logged in an unknown format and the conversion fails on an unrecognised record length, so it may
  be corrupted.

A fourth period is present but empty of gas fluxes: the LI-7200 was replaced on 11 January 2019 and
both units were defective, so the `2019_2` file contributes no `FC` and no `LE`. It does contribute
sensible heat flux, which comes from the sonic. Coverage below is therefore counted on `FC`, not on
a flux that survives a gas-analyser fault.

The 2025 run additionally exported six biomet inputs (`TA_1_1_1`, `RH_1_1_1`, `PA_1_1_1`,
`SW_IN_1_1_1`, `LW_IN_1_1_1`, `PPFD_IN_1_1_1`) that do not belong in the EC output and that no
earlier run carries. They are dropped after the merge, so the merged record has one column set over
its whole length; the meteo variables come from the `10_METEO` products, not from the EddyPro
output.

## 🕐 Timestamp convention

The EddyPro output carries `TIMESTAMP_START` and `TIMESTAMP_END`. `MultiDataFileReader` is called
with `output_middle_timestamp=True`, so the merged record is indexed on `TIMESTAMP_MIDDLE`, the same
convention the meteo products use. Timestamps are used as written by EddyPro; nothing is shifted
here.

## ⚙️ Settings

In [1]:
# --- Source ------------------------------------------------------------------------------------
# The Level-0 EddyPro FLUXNET output files. They live in the external (untracked) data folder and
# are addressed by absolute path, so the notebook does not depend on the working directory.
SOURCEDIR = (r"F:\Sync\luhk_work\dev-data\datasets-data\dataset_ch-lae_flux_product-data"
             r"\workflow\0_data\OPENLAG-IRGA72-Level-0_fluxnet_2016-2025")
SOURCEPATTERN = "*_adv.csv"  # the FLUXNET output files; anything else in that folder is ignored
FILETYPE = 'EDDYPRO-FLUXNET-CSV-30MIN'

# --- Output ------------------------------------------------------------------------------------
# Data files carry the same numeric prefix and the same relative path as the notebook producing
# them, in the external data folder. Same convention as the meteo notebooks.
OUTNAME = "01_OPENLAG_EDDYPRO_FLUXNET_OUTPUT_IRGA72_2016-2025"
OUTPATH = (r"F:\Sync\luhk_work\dev-data\datasets-data\dataset_ch-lae_flux_product-data"
           r"\workflow\00_L0_checks\IRGA72")

# --- Biomet columns exported by mistake ---------------------------------------------------------
# The 2025 run wrote its biomet inputs into the EC output. They belong to the meteo products in
# 10_METEO, not here, and only that one run carries them. Dropping them keeps the merged record on
# one column set; the drop is guarded below by a scan of the source file headers.
DROPCOLS = ['TA_1_1_1', 'RH_1_1_1', 'PA_1_1_1', 'SW_IN_1_1_1', 'LW_IN_1_1_1', 'PPFD_IN_1_1_1']
DROPCOLS_N_RUNS = 1  # how many runs are known to carry them; raise deliberately, never to pass

# --- Checks ------------------------------------------------------------------------------------
REQUIRED_TIME_RESOLUTION = '30min'
EXPECTED_YEARS = list(range(2016, 2026))  # every year the merged record must carry data for
CHECKCOL = 'FC'  # CO2 flux; used to check that each expected year actually contains data

In [2]:
from datetime import datetime
from pathlib import Path
import importlib.metadata

import pandas as pd

from diive.core.io.filereader import search_files, MultiDataFileReader
from diive.core.io.files import save_parquet, load_parquet

pd.set_option('display.max_rows', 30)
pd.set_option('display.max_columns', 30)
pd.set_option('display.width', 1000)

# Start time is kept so the last cell can report the total runtime of the notebook.
NOTEBOOK_START = datetime.now()
print(f"Last run: {NOTEBOOK_START.strftime('%Y-%m-%d %H:%M:%S')}")
print(f"diive version: v{importlib.metadata.version('diive')}")

Last run: 2026-07-31 12:42:40
diive version: v0.91.0


## 📂 Source files

In [3]:
sourcefiles = search_files(searchdirs=SOURCEDIR, pattern=SOURCEPATTERN)

# An empty result reads exactly like a folder that has not been updated yet, so it raises instead
# of merging nothing and writing an empty product.
assert len(sourcefiles) > 0, f"no files matching {SOURCEPATTERN} found in {SOURCEDIR}"

print(f"Found {len(sourcefiles)} source files in {SOURCEDIR}:")
for _f in sourcefiles:
    print(f"  {_f.name}")

Found 13 source files in F:\Sync\luhk_work\dev-data\datasets-data\dataset_ch-lae_flux_product-data\workflow\0_data\OPENLAG-IRGA72-Level-0_fluxnet_2016-2025:
  OPENLAG_2016_2+3_IRGA72_eddypro_CH-LAE_2016_2+3_FR-20250606-160443_fluxnet_2025-06-06T225430_adv.csv
  OPENLAG_2017_1+2_IRGA72_eddypro_CH-LAE_2017_1+2_FR-20250604-203500_fluxnet_2025-06-05T010037_adv.csv
  OPENLAG_2018_1_IRGA72_eddypro_CH-LAE_2018_1_FR-20250606-160435_fluxnet_2025-06-06T172838_adv.csv
  OPENLAG_2018_2+3+4_IRGA72_eddypro_CH-LAE_2018_2+3+4_FR-20250604-194123_fluxnet_2025-06-05T001439_adv.csv
  OPENLAG_2019_1_IRGA72_eddypro_CH-LAE_2019_1_FR-20250606-160428_fluxnet_2025-06-06T161009_adv.csv
  OPENLAG_2019_2_IRGA72_eddypro_CH-LAE_2019_2_FR-20250606-160405_fluxnet_2025-06-06T160657_adv.csv
  OPENLAG_2019_4_IRGA72_eddypro_CH-LAE_2019_4_FR-20250604-200113_fluxnet_2025-06-04T234306_adv.csv
  OPENLAG_2020_IRGA72_eddypro_CH-LAE_2020_FR-20250604-195655_fluxnet_2025-06-05T000703_adv.csv
  OPENLAG_2021_1_IRGA72_eddypro_CH-LAE_

## 🔗 Merge

In [4]:
%%time
d = MultiDataFileReader(filepaths=sourcefiles,
                        filetype=FILETYPE,
                        output_middle_timestamp=True)
df = d.data_df

> Reading file 
OPENLAG_2016_2+3_IRGA72_eddypro_CH-LAE_2016_2+3_FR-20250606-160443_fluxnet_2025-06-06T225430_adv.csv
...

F:\dev\diive\diive\core\io\filereader.py:591: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[_temp_parsed_index_col] = ''


> Reading file 
OPENLAG_2017_1+2_IRGA72_eddypro_CH-LAE_2017_1+2_FR-20250604-203500_fluxnet_2025-06-05T010037_adv.csv
...

F:\dev\diive\diive\core\io\filereader.py:591: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[_temp_parsed_index_col] = ''


> Reading file 
OPENLAG_2018_1_IRGA72_eddypro_CH-LAE_2018_1_FR-20250606-160435_fluxnet_2025-06-06T172838_adv.csv ...

...

> Reading file 
OPENLAG_2024_IRGA72_eddypro_CH-LAE_2024_FR-20250603-150844_fluxnet_2025-06-03T200327_adv.csv ...

F:\dev\diive\diive\core\io\filereader.py:591: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[_temp_parsed_index_col] = ''


> Reading file 
OPENLAG_2025_IRGA72_eddypro_CH-LAE_2025_1_FR-20260727-165458_fluxnet_2026-07-27T180908_adv.csv ...

F:\dev\diive\diive\core\io\filereader.py:591: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[_temp_parsed_index_col] = ''


CPU times: total: 1min 6s
Wall time: 1min 10s


## 🧹 Drop the biomet columns

The 2025 run exported the biomet inputs into the EC output by mistake. They are dropped here rather
than in each reader, so that the merged record has one column set over its whole length and nobody
downstream takes a meteo value from the EddyPro output instead of from the `10_METEO` products.

The drop is guarded by reading the header of every source file, which tests the cause rather than a
proxy for it. Guarding on the year does not work: a run begins at the last binary of the previous
December (`2025_1` starts at `2024123119.L00`), so its records reach back into 31 December 2024 and
the biomet columns legitimately hold data in two calendar years.

In [5]:
_present = [c for c in DROPCOLS if c in df.columns]

# Guard before the drop, on the cause rather than on a proxy for it: read the header of every
# source file and confirm that exactly one run carries these columns. Guarding on the year does not
# work here - a run starts at the last binary of the previous December (2025_1 starts at
# 2024123119.L00), so its records reach back into 31 December of the year before.
_files_with_biomet = []
for _f in sourcefiles:
    with open(_f, encoding='utf-8') as _fh:
        _header = _fh.readline().strip().split(',')
    if any(c in _header for c in DROPCOLS):
        _files_with_biomet.append(_f.name)

print("Source files carrying biomet columns:")
for _n in _files_with_biomet:
    print(f"  {_n}")
assert len(_files_with_biomet) == DROPCOLS_N_RUNS, \
    (f"{len(_files_with_biomet)} runs carry the biomet columns, expected {DROPCOLS_N_RUNS}. "
     f"Check whether the additional run really exported them by mistake before widening this.")

if _present:
    _biomet_records = df.index[df[_present].notna().any(axis=1)]
    print(f"\nColumns found: {_present}")
    print(f"Records holding data in them: {len(_biomet_records):,} "
          f"({_biomet_records[0]} -> {_biomet_records[-1]})")

df = df.drop(columns=_present)
print(f"\nDropped {len(_present)} column(s). {len(df.columns):,} columns remain.")

Source files carrying biomet columns:
  OPENLAG_2025_IRGA72_eddypro_CH-LAE_2025_1_FR-20260727-165458_fluxnet_2026-07-27T180908_adv.csv

Columns found: ['TA_1_1_1', 'RH_1_1_1', 'PA_1_1_1', 'SW_IN_1_1_1', 'LW_IN_1_1_1', 'PPFD_IN_1_1_1']
Records holding data in them: 17,530 (2024-12-31 19:15:00 -> 2025-12-31 23:45:00)

Dropped 6 column(s). 495 columns remain.


In [6]:
df

,AIR_MV,AIR_DENSITY,AIR_RHO_CP,AIR_CP,AOA_METHOD,AXES_ROTATION_METHOD,BOWEN,BURBA_METHOD,BADM_LOCATION_LAT,BADM_LOCATION_LONG,BADM_LOCATION_ELEV,BADM_HEIGHTC,BADM_INST_SAMPLING_INT,BADM_INST_AVERAGING_INT,BADM_INST_MODEL_SA,...,W_T_SONIC_COV_IBROM_N0004,W_NUM_SPIKES,WD_FILTER_NREX,W_SPIKE_NREX,W_ABSLIM_NREX,W_VM97_TEST,W_LGD,W_KID,W_ZCD,W_ITC,W_ITC_TEST,WBOOST_APPLIED,WPL_APPLIED,ZL,ZL_UNCORR
TIMESTAMP_MIDDLE,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
2016-01-11 15:15:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2016-01-11 15:45:00,0.025415,1.13966,1145.77,1005.36,0.0,1.0,NaN,0.0,47.4783,8.36439,689.0,37.0,20.0,30.0,NaN,...,NaN,1.0,0.0,1.0,0.0,801010000.0,1.50,7.65689,0.0,4.0,1.0,0.0,0.0,0.009173,0.009198
2016-01-11 16:15:00,0.025509,1.13546,1141.57,1005.38,0.0,1.0,NaN,0.0,47.4783,8.36439,689.0,37.0,20.0,30.0,NaN,...,NaN,0.0,0.0,0.0,0.0,801010000.0,1.55,7.31186,0.0,6.0,1.0,0.0,0.0,0.008916,0.008932
2016-01-11 16:45:00,0.025582,1.13222,1138.33,1005.40,0.0,1.0,NaN,0.0,47.4783,8.36439,689.0,37.0,20.0,30.0,NaN,...,NaN,0.0,0.0,0.0,0.0,800010000.0,0.10,8.96945,1.0,7.0,1.0,0.0,0.0,-0.024976,-0.025218
2016-01-11 17:15:00,0.025625,1.13030,1136.42,1005.41,0.0,1.0,NaN,0.0,47.4783,8.36439,689.0,37.0,20.0,30.0,NaN,...,NaN,1.0,0.0,2.0,0.0,800010000.0,0.80,7.41958,0.0,9.0,1.0,0.0,0.0,-0.000850,-0.000857
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2025-12-31 22:45:00,0.024009,1.20514,1213.13,1006.63,0.0,1.0,0.067569,0.0,47.4783,8.36439,689.0,37.0,20.0,30.0,NaN,...,NaN,0.0,0.0,0.0,0.0,800000011.0,0.00,11.29880,4601.0,49.0,3.0,0.0,0.0,-0.015730,-0.030361
2025-12-31 23:15:00,0.024015,1.20467,1212.83,1006.78,0.0,1.0,4.971290,0.0,47.4783,8.36439,689.0,37.0,20.0,30.0,NaN,...,NaN,18.0,0.0,32.0,0.0,800000011.0,0.00,19.45800,7892.0,32.0,3.0,0.0,0.0,0.686305,0.697002
2025-12-31 23:45:00,0.024015,1.20467,1212.86,1006.79,0.0,1.0,1.369620,0.0,47.4783,8.36439,689.0,37.0,20.0,30.0,NaN,...,NaN,1.0,0.0,2.0,0.0,800000011.0,0.00,13.98160,6271.0,20.0,2.0,0.0,0.0,2.106530,2.193120


In [7]:
print(f"Records: {len(df):,}")
print(f"Columns: {len(df.columns):,}")
print(f"Period:  {df.index[0]} -> {df.index[-1]}  (index: {df.index.name})")

Records: 174,836
Columns: 495
Period:  2016-01-11 15:15:00 -> 2026-01-01 00:45:00  (index: TIMESTAMP_MIDDLE)


## 🔍 Checks

In [8]:
# The merged index must be continuous 30MIN, unique and ascending. MultiDataFileReader reindexes
# to a continuous timestamp, so a failure here means the incoming files were not what was assumed.
assert df.index.is_monotonic_increasing, "timestamp index is not ascending"
assert not df.index.duplicated().any(), "duplicate timestamps in the merged record"
_steps = df.index.to_series().diff().dropna().unique()
assert len(_steps) == 1 and _steps[0] == pd.Timedelta(REQUIRED_TIME_RESOLUTION), \
    f"index is not continuous {REQUIRED_TIME_RESOLUTION}, found steps: {_steps}"

# A year can be present in the index and still be empty: the index is continuous, so a source file
# that was not exported shows up as a year full of NaN rather than as a gap in the index.
_available = df[CHECKCOL].groupby(df.index.year).count()
_missing = [y for y in EXPECTED_YEARS if _available.get(y, 0) == 0]
assert not _missing, f"no {CHECKCOL} data for {_missing}, source file missing?"

print(f"Available {CHECKCOL} records per year:")
display(_available.rename(f'{CHECKCOL} records'))

Available FC records per year:


TIMESTAMP_MIDDLE
2016    16378
2017    17066
2018    16823
2019    13602
2020    16126
2021    16303
2022    15225
2023    16879
2024    16943
2025    17404
2026        2
Name: FC records, dtype: int64

## 💾 Export

In [9]:
Path(OUTPATH).mkdir(parents=True, exist_ok=True)
filepath = save_parquet(filename=OUTNAME, data=df, outpath=OUTPATH)
df.to_csv(Path(OUTPATH) / f"{OUTNAME}.csv")
print(f"Saved to: {filepath}")

> Saved file 
F:\Sync\luhk_work\dev-data\datasets-data\dataset_ch-lae_flux_product-data\workflow\00_L0_checks\IRGA
72\01_OPENLAG_EDDYPRO_FLUXNET_OUTPUT_IRGA72_2016-2025.parquet (3.395 seconds).

Saved to: F:\Sync\luhk_work\dev-data\datasets-data\dataset_ch-lae_flux_product-data\workflow\00_L0_checks\IRGA72\01_OPENLAG_EDDYPRO_FLUXNET_OUTPUT_IRGA72_2016-2025.parquet


## ✅ Read the written file back

In [10]:
_written = load_parquet(filepath=str(Path(OUTPATH) / f"{OUTNAME}.parquet"))

print(f"index:   {_written.index[0]} -> {_written.index[-1]}  ({len(_written):,} records)")
print(f"columns: {len(_written.columns):,}")

assert len(_written) == len(df), "record count changed on write"
assert list(_written.columns) == list(df.columns), "columns changed on write"
assert _written[CHECKCOL].notna().sum() == df[CHECKCOL].notna().sum(), \
    f"{CHECKCOL} count changed on write"
print("\nWritten file verified.")

> Loaded .parquet file 
F:\Sync\luhk_work\dev-data\datasets-data\dataset_ch-lae_flux_product-data\workflow\00_L0_checks\IRGA
72\01_OPENLAG_EDDYPRO_FLUXNET_OUTPUT_IRGA72_2016-2025.parquet (1.319 seconds).

index:   2016-01-11 15:15:00 -> 2026-01-01 00:45:00  (174,836 records)
columns: 495

Written file verified.


***

# End of notebook.

In [11]:
NOTEBOOK_END = datetime.now()
_h, _rem = divmod(int((NOTEBOOK_END - NOTEBOOK_START).total_seconds()), 3600)
_m, _s = divmod(_rem, 60)

print(f"Started:  {NOTEBOOK_START.strftime('%Y-%m-%d %H:%M:%S')}")
print(f"Finished: {NOTEBOOK_END.strftime('%Y-%m-%d %H:%M:%S')}")
print(f"Runtime:  {_h:02d}:{_m:02d}:{_s:02d} (hh:mm:ss)")

Started:  2026-07-31 12:42:40
Finished: 2026-07-31 12:44:44
Runtime:  00:02:03 (hh:mm:ss)
